# 02a · La CA di bootstrap su DPS
*Blocco 2 · IoT Hub con ADR e DPS. Passo di creazione.*

Il device ha bisogno di una credenziale per presentarsi a DPS: un certificato firmato da una **CA di bootstrap**. Qui si creano le CA (una per flotta), si dà a chi esegue l'accesso ai dati di DPS e si **carica e verifica** ogni CA su DPS.
Non è la CA di ADR del notebook 01: quella emette il certificato operativo. Questa serve solo a presentare il device a DPS.

**Prima:** il notebook 01. **Dopo:** il notebook 02b crea i gruppi di registrazione.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Fabric: valori dalla Eventstream creata dal 17a (417bd75b-4747-4eff-9d6e-d00ca5964001)

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## 1 · Una CA di bootstrap per flotta
Il device si presenta a DPS con un certificato firmato da una CA di bootstrap; DPS ritrova il **gruppo di registrazione**, e quindi l'**Hub**, da quella CA.
Due flotte, due gruppi, due Hub: servono **due CA**, perché un certificato può stare **in un solo gruppo** di registrazione dello stesso DPS (Microsoft Learn, *X.509 certificate attestation*).
Un'alternativa è una Root sola con una CA intermedia per flotta. Qui, per tenere la demo semplice, sono due CA indipendenti.
Le chiavi private (`ca-key.pem`) restano su questo computer: proteggile.

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
foreach ($h in $hubs) {
    $ca = New-DpsBootstrapCa -Group $h.Group -OutDir $out
    $stato = if ($ca.Created) { 'creata' } else { "gia' esistente" }
    "$($h.Group): CA $stato in $($ca.Cert)"
}

robots: CA gia' esistente in D:\Dev\2026\IoTHub\WPC2026\artifacts\bootstrap-ca-robots\ca.pem

trucks: CA gia' esistente in D:\Dev\2026\IoTHub\WPC2026\artifacts\bootstrap-ca-trucks\ca.pem

## 2 · Accesso ai dati di DPS
**Owner non basta**: servono i ruoli data-plane. Assegniamo *Device Provisioning Service Data Contributor* solo su questo DPS e
aspettiamo che il ruolo si propaghi.

In [ ]:
$me = az ad signed-in-user show --query id -o tsv --only-show-errors
$dpsId = az iot dps show --name $dps.Name --resource-group $rg --query id -o tsv --only-show-errors
az role assignment create --assignee-object-id $me --assignee-principal-type User --role dfce44e4-17b7-4bd1-a6d1-04996ec95633 --scope $dpsId -o none --only-show-errors

for ($i = 1; $i -le 12; $i++) {
    az iot dps enrollment-group list --dps-name $dps.Name --resource-group $rg --auth-type login --query '[].enrollmentGroupId' -o none --only-show-errors 2>$null
    if ($LASTEXITCODE -eq 0) { "accesso data-plane verificato."; break }
    "attendo la propagazione dei ruoli ($i/12)..."
    Start-Sleep -Seconds 10
}

accesso data-plane verificato.

## 3 · La CA su DPS: carica e **verifica**
La trappola: creando il gruppo con `--certificate-path` DPS tratta la CA come *intermedia* e rifiuta ogni device con
`401002 Invalid certificate` (con ECC P-384, P-256 e RSA). La CA va **caricata e verificata** con la *proof of possession*:
DPS genera un codice, si firma con la CA un certificato con CN = codice, DPS lo verifica.

In [ ]:
foreach ($h in $hubs) {
    "--- CA pcert-$workload-$stage-$($h.Group)"
    Register-DemoDpsCa -DpsName $dps.Name -Rg $rg -Name "pcert-$workload-$stage-$($h.Group)" -CaDir (Join-Path $out "bootstrap-ca-$($h.Group)")
}

  CA 'pcert-iot26-dev-robots' gia' verificata su DPS.

  CA 'pcert-iot26-dev-trucks' gia' verificata su DPS.

--- CA pcert-iot26-dev-robots

--- CA pcert-iot26-dev-trucks

Prossimo passo: **02b**, i gruppi di registrazione che usano queste CA.